# Brand Monitoring with SerpApi

This notebook walks through how to use SerpApi to monitor brand mentions across Google News, YouTube, and Google Perspectives (Reddit, LinkedIn, Quora). We'll use `serpapi` as our search term.

## Setup

Install dependencies:
```bash
pip install serpapi pandas altair
```

You'll need a SerpApi API key. Get one at https://serpapi.com/manage-api-key, then set it as an environment variable:
```bash
export SERPAPI_KEY=your_key_here
```

In [ ]:
import os
import re
from datetime import datetime, timedelta, timezone

import altair as alt
import pandas as pd
import serpapi

SERPAPI_KEY = os.environ.get("SERPAPI_KEY", "")

# Change this to monitor a different brand
BRAND = "serpapi"

client = serpapi.Client(api_key=SERPAPI_KEY)

## Relative Date Parsing

SerpApi returns dates as relative strings like `"2 hours ago"` or `"5 days ago"`. We need a helper to convert these into proper `datetime` objects for sorting and grouping.

In [15]:
RELATIVE_DATE_RE = re.compile(
    r"(\d+)\s+(second|minute|hour|day|week|month|year)s?\s+ago", re.IGNORECASE
)

UNIT_TO_TIMEDELTA = {
    "second": lambda n: timedelta(seconds=n),
    "minute": lambda n: timedelta(minutes=n),
    "hour": lambda n: timedelta(hours=n),
    "day": lambda n: timedelta(days=n),
    "week": lambda n: timedelta(weeks=n),
    "month": lambda n: timedelta(days=n * 30),
    "year": lambda n: timedelta(days=n * 365),
}


def parse_relative_date(text):
    """Convert relative date strings like '2 hours ago' into datetime objects."""
    if not text:
        return datetime.now(timezone.utc)

    match = RELATIVE_DATE_RE.search(str(text))
    if not match:
        return datetime.now(timezone.utc)

    amount = int(match.group(1))
    unit = match.group(2).lower()
    delta = UNIT_TO_TIMEDELTA.get(unit, lambda n: timedelta())(amount)

    return datetime.now(timezone.utc) - delta


# Quick test
print(parse_relative_date("2 hours ago"))
print(parse_relative_date("5 days ago"))

2026-04-06 17:41:47.877229+00:00
2026-04-01 19:41:47.877326+00:00


## 1. Google News

The Google News engine (`engine: "google_news"`) returns articles from news sources. The response key is `news_results`, where each item includes the article title, source, date, and snippet.

In [16]:
def fetch_news(client, brand):
    """Fetch news articles mentioning the brand via Google News."""
    results = client.search({
        "engine": "google_news",
        "q": brand,
        "gl": "us",
        "hl": "en",
    })
    return results.get("news_results", [])


raw_news = fetch_news(client, BRAND)
print(f"Fetched {len(raw_news)} news articles for '{BRAND}'")

Fetched 78 news articles for 'serpapi'


In [17]:
# Inspect the first news result to understand the structure
if raw_news:
    sample = raw_news[0]
    print(f"Title: {sample.get('title')}")
    print(f"Source: {sample.get('source')}")
    print(f"Date: {sample.get('date')}")
    print(f"Link: {sample.get('link')}")

Title: Why we’re taking legal action against SerpApi’s unlawful scraping
Source: {'name': 'blog.google', 'icon': 'https://encrypted-tbn1.gstatic.com/faviconV2?url=https://blog.google&client=NEWS_360&size=96&type=FAVICON&fallback_opts=TYPE,SIZE,URL'}
Date: 12/19/2025, 08:00 AM, +0000 UTC
Link: https://blog.google/innovation-and-ai/technology/safety-security/serpapi-lawsuit/


### Transforming News Results

We extract the source name (which can be a dict or a plain string depending on the result), parse the relative date, and build a flat record for each article.

In [18]:
def transform_news(results):
    """Convert raw Google News results into structured records."""
    records = []
    for item in results:
        source = item.get("source") or {}
        source_name = source.get("name", "Unknown") if isinstance(source, dict) else str(source)

        records.append({
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "source": source_name,
            "date": parse_relative_date(item.get("date", "")),
            "snippet": item.get("snippet", ""),
        })
    return records


news_records = transform_news(raw_news)
news_df = pd.DataFrame(news_records)
news_df.head(10)

,title,link,source,date,snippet
0,Why we’re taking legal action against SerpApi’...,https://blog.google/innovation-and-ai/technolo...,blog.google,2026-04-06 19:41:48.508947+00:00,
1,Google Sues SerpApi for ‘Parasitic’ Scraping a...,https://ipwatchdog.com/2025/12/26/google-sues-...,IPWatchdog.com,2026-04-06 19:41:48.508953+00:00,
2,SerpApi fights back against Google lawsuit,https://www.computerworld.com/article/4136288/...,Computerworld,2026-04-06 19:41:48.508957+00:00,
3,Google Sued SerpApi Over Scraping Search Results,https://www.seroundtable.com/google-sues-serpa...,Search Engine Roundtable,2026-04-06 19:41:48.508959+00:00,
4,SerpApi Files Motion to Dismiss Google's Compl...,https://www.prnewswire.com/news-releases/serpa...,PR Newswire,2026-04-06 19:41:48.508962+00:00,
5,SerpApi moves to dismiss Google scraping lawsuit,https://searchengineland.com/serpapi-motion-di...,Search Engine Land,2026-04-06 19:41:48.508964+00:00,
6,Google lobs lawsuit at search result scraping ...,https://arstechnica.com/google/2025/12/google-...,Ars Technica,2026-04-06 19:41:48.508967+00:00,
7,SerpApi asks court to dismiss Google web scrap...,https://www.theregister.com/2026/02/21/serpapi...,theregister.com,2026-04-06 19:41:48.508969+00:00,
8,SerpApi Fires Back: Google Built Empire on Scr...,https://www.techbuzz.ai/articles/serpapi-fires...,The Tech Buzz,2026-04-06 19:41:48.508971+00:00,
9,Google Says SerpApi Bypasses Security To Scrap...,https://www.law360.com/articles/2424443/google...,Law360,2026-04-06 19:41:48.508974+00:00,


In [19]:
source_df = news_df["source"].value_counts().head(10).reset_index()
source_df.columns = ["source", "count"]

alt.Chart(source_df).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("count:Q", title="Articles"),
    y=alt.Y("source:N", sort="-x", title=""),
    color=alt.value("#4A90D9"),
    tooltip=["source:N", "count:Q"],
).properties(width=500, height=350, title="Top 10 News Sources")

alt.Chart(...)

## 2. YouTube

The YouTube engine (`engine: "youtube"`) uses `search_query` instead of `q`. We apply two time filters — week and month — and deduplicate by video link, since the month filter includes the week's results.

In [20]:
YT_FILTER_WEEK = "EgIIAw%3D%3D"
YT_FILTER_MONTH = "EgIIBA%3D%3D"


def fetch_youtube(client, brand):
    """Fetch YouTube videos mentioning the brand, combining week and month filters."""
    seen = set()
    videos = []

    for sp_filter in (YT_FILTER_WEEK, YT_FILTER_MONTH):
        results = client.search({
            "engine": "youtube",
            "search_query": brand,
            "sp": sp_filter,
        })
        for video in results.get("video_results", []):
            link = video.get("link", "")
            if link and link not in seen:
                seen.add(link)
                videos.append(video)

    return videos


raw_youtube = fetch_youtube(client, BRAND)
print(f"Fetched {len(raw_youtube)} YouTube videos for '{BRAND}'")

Fetched 33 YouTube videos for 'serpapi'


In [21]:
def transform_youtube(results):
    """Convert raw YouTube results into structured records."""
    records = []
    for item in results:
        channel = item.get("channel") or {}
        channel_name = channel.get("name", "Unknown") if isinstance(channel, dict) else str(channel)

        views = item.get("views") or 0
        if isinstance(views, str):
            views = int(re.sub(r"[^\d]", "", views) or 0)
        views = int(views)

        records.append({
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "channel": channel_name,
            "views": views,
            "date": parse_relative_date(item.get("published_date", "")),
            "length": item.get("length", ""),
        })
    return records


yt_records = transform_youtube(raw_youtube)
yt_df = pd.DataFrame(yt_records)
yt_df.head(10)

,title,link,channel,views,date,length
0,Build an AI Agent with Real Time Data Next js ...,https://www.youtube.com/watch?v=cihIzbG7z4A,Cand Dev,89,2026-04-06 10:41:49.201866+00:00,15:41
1,Build Your Own AI Social Listening Tool with P...,https://www.youtube.com/watch?v=pGd-OxrZlbM,HasData – Scraping APIs & No-Code Scrapers,6,2026-04-06 08:41:49.201881+00:00,7:29
2,"🔴 Open Source Work, Chat and Q&A",https://www.youtube.com/watch?v=aPzv31j4Yks,nunomaduro,0,2026-04-06 19:41:49.201885+00:00,
3,Automatisez votre veille d'actualités hebdo av...,https://www.youtube.com/watch?v=QHvfykuSbF8,Growth AI,6,2026-04-01 19:41:49.201891+00:00,5:16
4,Building Real-Time AI Search Agents with N8N,https://www.youtube.com/watch?v=JeXYIfqtzGM,Garima Garg,3,2026-03-30 19:41:49.201895+00:00,1:54
5,How 10x Developers Actually Use AI,https://www.youtube.com/watch?v=_78-xwTyQeE,nunomaduro and Aaron Francis,4713,2026-03-30 19:41:49.201900+00:00,10:29
6,🔴 JetBrains Air: Agentic Development,https://www.youtube.com/watch?v=BgkpOxYNhOU,nunomaduro,1521,2026-04-01 19:41:49.201903+00:00,2:08:55
7,I Built $1 BILLION SaaS Product In 24 Hrs as a...,https://www.youtube.com/watch?v=avf18yzj_0I,CK Data Tech,23,2026-04-06 05:41:49.201907+00:00,26:04
8,🔴 Learning TanStack,https://www.youtube.com/watch?v=f8gxjveZ9QI,nunomaduro,1769,2026-04-04 19:41:49.201911+00:00,1:55:38
9,Check Best API Search Homepage,https://www.youtube.com/watch?v=euOAULV7s-M,Portillos menu,13,2026-03-30 19:41:49.201914+00:00,4:40


In [22]:
channel_df = yt_df.groupby("channel")["views"].sum().reset_index()
channel_df = channel_df.sort_values("views", ascending=False).head(10)

alt.Chart(channel_df).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("views:Q", title="Views", axis=alt.Axis(format="~s")),
    y=alt.Y("channel:N", sort="-x", title=""),
    color=alt.value("#4A90D9"),
    tooltip=["channel:N", alt.Tooltip("views:Q", format=",")],
).properties(width=500, height=350, title="Views by Channel (Top 10)")

alt.Chart(...)

## 3. Google Perspectives

Google Perspectives surfaces user-generated content from platforms like Reddit, LinkedIn, Quora, and blogs. It appears as a section within standard Google search results (`engine: "google"`, response key: `perspectives`).

In [23]:
def fetch_perspectives(client, brand):
    """Fetch user-generated content (Reddit, LinkedIn, Quora) via Google Perspectives."""
    results = client.search({
        "engine": "google",
        "q": brand,
        "google_domain": "google.com",
    })
    return results.get("perspectives", [])


raw_perspectives = fetch_perspectives(client, BRAND)
print(f"Fetched {len(raw_perspectives)} perspectives for '{BRAND}'")

Fetched 6 perspectives for 'serpapi'


In [24]:
def transform_perspectives(results):
    """Convert raw Google Perspectives results into structured records."""
    records = []
    for item in results:
        records.append({
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "source": item.get("source") or "Unknown",
            "author": item.get("author") or "",
            "date": parse_relative_date(item.get("date", "")),
            "snippet": item.get("snippet", ""),
        })
    return records


persp_records = transform_perspectives(raw_perspectives)
persp_df = pd.DataFrame(persp_records)
persp_df.head(10)

,title,link,source,author,date,snippet
0,Want public web data without scraper headaches...,https://www.linkedin.com/posts/nk-systemdesign...,LinkedIn,Neo Kim,2026-03-07 19:41:49.560818+00:00,
1,Free/Cheap SERP API to get google search trends,https://www.reddit.com/r/TechSEO/comments/1rjg...,Reddit,r/TechSEO,2026-03-07 19:41:49.560829+00:00,Popular comment · Why not use the Google Trend...
2,"State of SERP, Q1 2026. A lot changed since Q4...",https://www.linkedin.com/posts/jasongrad_state...,LinkedIn,Jason Grad,2026-03-23 19:41:49.560835+00:00,
3,Build an AI Agent with Real Time Data Next js ...,https://www.youtube.com/watch?v=cihIzbG7z4A,YouTube,Cand Dev,2026-04-06 10:41:49.560839+00:00,
4,Google's recent lawsuit against SerpAPI reveal...,https://www.linkedin.com/posts/resoneo_googles...,LinkedIn,Olivier de Segonzac,2026-04-06 19:41:49.560842+00:00,
5,Best Serp API tool for B2B data collection,https://www.reddit.com/r/SaaSSales/comments/1q...,Reddit,r/SaaSSales,2026-04-06 19:41:49.560843+00:00,Popular comment · I been using scrapfly as my ...


In [25]:
if not persp_df.empty:
    platform_df = persp_df["source"].value_counts().reset_index()
    platform_df.columns = ["source", "count"]

    chart = alt.Chart(platform_df).mark_arc(innerRadius=60, outerRadius=120).encode(
        theta=alt.Theta("count:Q"),
        color=alt.Color("source:N", legend=alt.Legend(title="Platform")),
        tooltip=["source:N", "count:Q"],
    ).properties(width=400, height=350, title="Mentions by Platform")

    display(chart)
else:
    print("No perspectives found for this brand.")

alt.Chart(...)

## Combined Overview

Let's summarize all brand mentions across the three sources.

In [26]:
total = len(news_records) + len(yt_records) + len(persp_records)

print(f"Total Mentions: {total}")
print(f"  News Articles: {len(news_records)}")
print(f"  YouTube Videos: {len(yt_records)}")
print(f"  Perspectives: {len(persp_records)}")

if not news_df.empty:
    print(f"\nNews — Unique Sources: {news_df['source'].nunique()}")
if not yt_df.empty:
    print(f"YouTube — Unique Channels: {yt_df['channel'].nunique()}, Total Views: {yt_df['views'].sum():,}")
if not persp_df.empty:
    print(f"Perspectives — Platforms: {persp_df['source'].nunique()}")

Total Mentions: 117
  News Articles: 78
  YouTube Videos: 33
  Perspectives: 6

News — Unique Sources: 52
YouTube — Unique Channels: 26, Total Views: 1,116,119
Perspectives — Platforms: 3
